In [1]:

import os
import gc
import tempfile
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import KDTree
from tqdm import tqdm
import psutil
import pytz


In [2]:
# -*- coding: utf-8 -*-
"""
Unir meteo (Open-Meteo por celdas de rejilla) con accidentes/no-accidentes por (cluster_id, time_n):
- Convierte tiempos a Europe/Madrid correctamente (Open-Meteo ya viene en local).
- Precalcula una sola vez KDTree sobre las celdas únicas del CSV meteo.
- Mapea cada cluster a su (lat_cell, lon_cell) más cercano.
- Agrega meteo por (cluster_id, time_n) en streaming (chunks).
- Merge final por (cluster_id, time_n) con opción de hacer merge por chunks si falta memoria.

Requisitos: pandas, geopandas, numpy, scipy, pytz, tqdm, psutil
"""

# ---------------------------
# Parámetros de entrada/salida
# ---------------------------
INPUT_GEOJSON   = "../data/processed/full_accidents_non_accidents.geojson"
INPUT_CENTROIDS = "../data/processed/centroides_validos.csv"
INPUT_METEO_CSV = "../data/processed/meteo-clusters.csv"        # generado desde Open-Meteo
OUTPUT_GEOJSON  = "../data/processed/dataset_accidents_weather_merged.geojson"

# Opciones de rendimiento
READ_CHUNK_METEO = 200_000     # tamaño de chunk para leer meteo
READ_CHUNK_ACC   = 1_000_000   # tamaño de chunk para merge final (si activas MERGE_IN_CHUNKS)
MERGE_IN_CHUNKS  = True        # True = merge por chunks (más seguro en memoria)

TZ = "Europe/Madrid"
MET_COLS = ["temperature_2m (°C)", "precipitation (mm)", "rain (mm)", "snowfall (cm)", "wind_speed_10m (km/h)"]

# ---------------------------
# Utilidades
# ---------------------------
def print_mem(note=""):
    rss = psutil.Process().memory_info().rss / 1024**2
    print(f"[MEM] {note} -> {rss:.2f} MB")

def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype("float32")
    # cuidado con categorizar IDs; mantenemos Int64 nullable
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = df[col].astype("Int64", errors="ignore")
    return df

def to_local_from_naive(series: pd.Series) -> pd.Series:
    """
    Open-Meteo con timezone=Europe/Madrid devuelve strings NAIVE en hora local.
    Localizamos como Europe/Madrid (sin convertir) y alineamos a inicio de la hora.
    """
    dt = pd.to_datetime(series, errors="coerce")  # naive local
    dt = dt.dt.tz_localize(TZ, ambiguous="infer", nonexistent="shift_forward")
    return dt.dt.floor("h")

def ensure_local(series: pd.Series) -> pd.Series:
    """
    Garantiza tz-aware Europe/Madrid y floor("h") para series que pueden venir naive o con tz.
    Úsalo para df_acc_full['time_n'] y para columnas ya tz-aware.
    """
    dt = pd.to_datetime(series, errors="coerce")
    if getattr(dt.dtype, "tz", None) is None:
        dt = dt.dt.tz_localize(TZ, ambiguous="infer", nonexistent="shift_forward")
    else:
        dt = dt.dt.tz_convert(TZ)
    return dt.dt.floor("h")

# ---------------------------
# 1) Cargar accidentes/no-accidentes
# ---------------------------
print("Cargando accidentes/no-accidentes…")
gdf_acc = gpd.read_file(INPUT_GEOJSON)
print_mem("tras leer geojson")

# Asegurar time_n y cluster_id
gdf_acc["time_n"] = ensure_local(gdf_acc["time_n"])
gdf_acc["cluster_id"] = pd.to_numeric(gdf_acc["cluster_id"], errors="coerce").astype("Int64")

# Coordenadas por si las necesitas
gdf_acc["Longitud"] = gdf_acc.geometry.x
gdf_acc["Latitud"]  = gdf_acc.geometry.y

gdf_acc = gdf_acc[["cluster_id", "time_n", "geometry"] + [c for c in gdf_acc.columns if c not in ("cluster_id","time_n","geometry")]]
print(f"Registros en accidentes: {len(gdf_acc):,}")
print(f"Rango time_n: {gdf_acc['time_n'].min()} → {gdf_acc['time_n'].max()}")
print_mem("accidentes preparados")

# ---------------------------
# 2) Cargar centroides válidos
# ---------------------------
print("Cargando centroides válidos…")
centroids = pd.read_csv(INPUT_CENTROIDS)
centroids["cluster_id"] = pd.to_numeric(centroids["cluster_id"], errors="coerce").astype("Int64")
centroids["Latitud_centroide"]  = pd.to_numeric(centroids["Latitud_centroide"], errors="coerce")
centroids["Longitud_centroide"] = pd.to_numeric(centroids["Longitud_centroide"], errors="coerce")

# Quitar duplicados de cluster_id (nos quedamos con el primero)
dups = centroids[centroids["cluster_id"].duplicated(keep=False)]
if not dups.empty:
    print(f"⚠️ Duplicados en centroides por cluster_id: {len(dups)} → se quedará el primero")
centroids = centroids.drop_duplicates(subset=["cluster_id"], keep="first")

centroids = centroids.dropna(subset=["cluster_id","Latitud_centroide","Longitud_centroide"])
centroids = optimize_dtypes(centroids)
print(f"Clusters válidos en centroides: {len(centroids)}")
print_mem("centroides preparados")

# ---------------------------
# 3) Celdas únicas del CSV de meteo (rejilla del modelo)
# ---------------------------
print("Leyendo coordenadas únicas del CSV meteo…")
meteo_tiles = pd.read_csv(INPUT_METEO_CSV, usecols=["latitude","longitude"])
meteo_tiles = meteo_tiles.drop_duplicates().reset_index(drop=True)
meteo_tiles["latitude"]  = meteo_tiles["latitude"].astype("float32")
meteo_tiles["longitude"] = meteo_tiles["longitude"].astype("float32")
print(f"Celdas únicas en meteo: {len(meteo_tiles)}")
print_mem("tiles meteo")

# ---------------------------
# 4) KDTree UNA VEZ: cluster → celda (lat_cell, lon_cell)
# ---------------------------
print("Construyendo KDTree (tiles meteo)…")
tile_tree = KDTree(meteo_tiles[["latitude","longitude"]].values)

centroid_coords = centroids[["Latitud_centroide","Longitud_centroide"]].to_numpy()
_, idx = tile_tree.query(centroid_coords)

centroids["lat_cell"] = meteo_tiles.iloc[idx]["latitude"].values
centroids["lon_cell"] = meteo_tiles.iloc[idx]["longitude"].values

cluster_to_tile = centroids[["cluster_id","lat_cell","lon_cell"]].drop_duplicates()
print("Ejemplo mapeo cluster→tile:")
print(cluster_to_tile.head())
print_mem("KDTree mapeado")

# ---------------------------
# 5) Agregar meteo por (cluster_id, time_n) leyendo por chunks
# ---------------------------
print("Agregando meteo por (cluster_id, time_n) en streaming…")
agg_parts = []
reader = pd.read_csv(INPUT_METEO_CSV, chunksize=READ_CHUNK_METEO)

for i, ch in enumerate(tqdm(reader, desc="Chunks meteo")):
    # time local (naive) → tz-aware Europe/Madrid
    ch["time_n"] = to_local_from_naive(ch["time"])
    ch = ch.drop(columns=["time"], errors="ignore")

    # tipos
    ch["latitude"]  = ch["latitude"].astype("float32")
    ch["longitude"] = ch["longitude"].astype("float32")

    # JOIN a tile → cluster_id
    ch = ch.merge(
        cluster_to_tile,
        left_on=["latitude","longitude"],
        right_on=["lat_cell","lon_cell"],
        how="left"
    )

    # Agregar (media) por (cluster_id, time_n)
    ch = ch.dropna(subset=["cluster_id","time_n"])
    ch_agg = (ch.groupby(["cluster_id","time_n"], as_index=False)[MET_COLS].mean())

    agg_parts.append(ch_agg)

    # liberar
    del ch, ch_agg
    if i % 5 == 0:
        gc.collect()
        print_mem(f"tras chunk meteo #{i}")

# Concatenar y segunda agregación por seguridad
meteo_agg = pd.concat(agg_parts, ignore_index=True) if agg_parts else pd.DataFrame(columns=["cluster_id","time_n"]+MET_COLS)
meteo_agg["cluster_id"] = pd.to_numeric(meteo_agg["cluster_id"], errors="coerce").astype("Int64")
meteo_agg["time_n"] = ensure_local(meteo_agg["time_n"])  # por si acaso
meteo_agg = (meteo_agg.groupby(["cluster_id","time_n"], as_index=False)[MET_COLS].mean())
meteo_agg = optimize_dtypes(meteo_agg)

print(f"Meteo agregada: {len(meteo_agg):,} filas | rango {meteo_agg['time_n'].min()} → {meteo_agg['time_n'].max()}")
print_mem("meteo agregada")

# ---------------------------
# 6) Merge final con accidentes
# ---------------------------
print("Uniendo con accidentes…")
temp_dir = tempfile.mkdtemp()
tmp_files = []

if MERGE_IN_CHUNKS:
    n = len(gdf_acc)
    n_chunks = (n // READ_CHUNK_ACC) + 1
    print(f"Merge por chunks: {n_chunks} trozos de hasta {READ_CHUNK_ACC:,} filas")

    for k in tqdm(range(n_chunks), desc="Chunks merge"):
        i0 = k * READ_CHUNK_ACC
        i1 = min((k+1) * READ_CHUNK_ACC, n)
        part = gdf_acc.iloc[i0:i1][["cluster_id","time_n","geometry"]].copy()

        merged = part.merge(
            meteo_agg,
            on=["cluster_id","time_n"],
            how="left",
            validate="m:1"
        )

        # imputar opcional precip/snow/rain a 0
        for c in ["precipitation (mm)", "snowfall (cm)", "rain (mm)"]:
            if c in merged.columns:
                merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0).astype("float32")

        fpath = os.path.join(temp_dir, f"merged_{k}.parquet")
        merged.to_parquet(fpath, index=False)
        tmp_files.append(fpath)

        del part, merged
        gc.collect()
        print_mem(f"tras chunk merge #{k}")

    # Juntar todo y guardar GeoJSON
    print("Concatenando todos los trozos y guardando…")
    out_frames = []
    for f in tqdm(tmp_files, desc="Concat final"):
        out_frames.append(pd.read_parquet(f))
    df_out = pd.concat(out_frames, ignore_index=True)
    # reconstruir GeoDataFrame
    gdf_out = gpd.GeoDataFrame(df_out, geometry="geometry", crs="EPSG:4326")
    gdf_out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

    # limpiar temporales
    for f in tmp_files:
        try: os.remove(f)
        except: pass
    try: os.rmdir(temp_dir)
    except: pass

else:
    merged = gdf_acc.merge(
        meteo_agg,
        on=["cluster_id","time_n"],
        how="left",
        validate="m:1"
    )
    for c in ["precipitation (mm)", "snowfall (cm)", "rain (mm)"]:
        if c in merged.columns:
            merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0).astype("float32")

    gdf_out = gpd.GeoDataFrame(merged, geometry="geometry", crs="EPSG:4326")
    gdf_out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

print(f"✅ Guardado: {OUTPUT_GEOJSON}")
print_mem("fin")


Cargando accidentes/no-accidentes…
[MEM] tras leer geojson -> 1965.58 MB


AmbiguousTimeError: 2019-10-27 02:00:00

In [4]:
df_acc_full = gpd.read_file("../data/processed/full_accidents_non_accidents.geojson")

In [3]:
gc.collect()

0

In [ ]:
df1 = pd.read_csv('../data/processed/df_accidents_weather_merged.csv')